# Step 1 - Data Load

In [1]:
import pandas as pd

In [2]:
import numpy as np

In [22]:
orignal_df = pd.read_csv("car_dataset.csv")

In [23]:
orignal_df.head()

,car_id,name,brand,category,pricePerDay
0,1,Toyota Fortuner,Toyota,SUV,85
1,2,Honda CR-V,Honda,SUV,75
2,3,Ford Explorer,Ford,SUV,90
3,4,Hyundai Tucson,Hyundai,SUV,70
4,5,Kia Sportage,Kia,SUV,68


In [24]:
feature_df  = pd.read_csv("feature_matrix.csv")

In [25]:
feature_df.head()

,2-Seater,Electric,Luxury,SUV,price_scaled,car_id,name
0,0.0,0.0,0.0,1.0,0.015018,1,Toyota Fortuner
1,0.0,0.0,0.0,1.0,0.006184,2,Honda CR-V
2,0.0,0.0,0.0,1.0,0.019435,3,Ford Explorer
3,0.0,0.0,0.0,1.0,0.001767,4,Hyundai Tucson
4,0.0,0.0,0.0,1.0,0.000000,5,Kia Sportage


# Step 2 - Recommendation Function

In [17]:
from sklearn.preprocessing import OneHotEncoder , MinMaxScaler

In [18]:
from sklearn.metrics.pairwise import cosine_similarity

In [45]:
def recommend(category , max_price , top_car = 5) :
    # filter the data from data set what user search 
    filtered = orignal_df[(orignal_df["category"] == category) & (orignal_df["pricePerDay"] <= max_price)]
    print(f"filtered cars : {filtered} \n\n\n")
    
    # check something find or not
    if filtered.empty:
        return {"message": "No cars found", "car_ids": []}


    # filter cars by id
    filtered_ids = filtered["car_id"].values
    print(f"Filtered car_ids: {filtered_ids}\n\n\n")

    # filetr car with id from feature_matrix dataset
    filter_feature = feature_df[feature_df["car_id"].isin(filtered_ids)]
    print(f"Filtered from feature matrix df: {filter_feature}\n\n\n")
    
        
    # selectiong the feature cols
    category_cols = [col for col in feature_df.columns if col not in ['car_id', 'name','pricePerDay', 'price_scaled', 'category']]

    feature_cols = category_cols + ["price_scaled"]

    # taking the Similarity
    features = filter_feature[feature_cols].values
    print(f"Taking the features : {features}\n\n\n")
    samilarity = cosine_similarity(features)

    # top n cars 
    first_car_similarity = samilarity[0]

    similar_cars = sorted(enumerate(first_car_similarity) , key=lambda x:x[1] , reverse=True)[1: top_car+1]

    # returen car id

    recommended_ids = []

    for i in similar_cars:
        car_index = i[0]
        car_id = filter_feature.iloc[car_index]["car_id"]
        recommended_ids.append(int(car_id))

    return {
    "category": category,
    "max_price": max_price,
    "car_ids": recommended_ids
    }
    

In [46]:
print(recommend("SUV", 100))

filtered cars :     car_id               name       brand category  pricePerDay
0        1    Toyota Fortuner      Toyota      SUV           85
1        2         Honda CR-V       Honda      SUV           75
2        3      Ford Explorer        Ford      SUV           90
3        4     Hyundai Tucson     Hyundai      SUV           70
4        5       Kia Sportage         Kia      SUV           68
6        7  Mitsubishi Pajero  Mitsubishi      SUV           95
40      41        Toyota RAV4      Toyota      SUV           72
41      42        Honda Pilot       Honda      SUV           88 



Filtered car_ids: [ 1  2  3  4  5  7 41 42]



Filtered from feature matrix df:     2-Seater  Electric  Luxury  SUV  price_scaled  car_id               name
0        0.0       0.0     0.0  1.0      0.015018       1    Toyota Fortuner
1        0.0       0.0     0.0  1.0      0.006184       2         Honda CR-V
2        0.0       0.0     0.0  1.0      0.019435       3      Ford Explorer
3        0.0    

In [47]:
print(recommend("SUV", 50))

filtered cars : Empty DataFrame
Columns: [car_id, name, brand, category, pricePerDay]
Index: [] 



{'message': 'No cars found', 'car_ids': []}


# Save the model 

In [48]:
import pickle

In [50]:
pickle.dump(orignal_df.to_dict() ,open("model/orignal_df.plk" , "wb"))

In [52]:
pickle.dump(feature_df.to_dict() , open("model/feature_df.plk" , "wb"))